# **📈 Stock Data Pipeline: From JSON Ingestion to SQL Analytics**

## **I. Introduction**

### **i. Problem Statement**

Candlestick charts are widely used in stock trading because they allow experienced traders to visually interpret price movement, identify pricing patterns, and support data-driven decision-making. However, for beginners, visual inspection can be incomplete or inconsistent, making it easy to miss important chart shapes, price patterns, or underlying market signals.

Building a stock price reader program adds structure and reliability to this process by transforming raw stock data into a standardized analytical workflow. This raises several important questions: Where can reliable stock data be sourced? How should raw stock information be processed, cleaned, and standardized? And can the workflow be designed to support reusable, large-scale stock data processing across multiple tickers?

### **ii. Project Outline**

In this project, I built a reusable data pipeline for sourcing stock price data, validating data quality, and storing the cleaned dataset in a SQL database for downstream analysis. The workflow is designed to standardize stock information processing and support repeatable analysis across multiple tickers.

The rest of this project follows three main sections:

1. JSON data ingestion and processing  
2. SQL data conversion and analysis  
3. Project summary and future roadmap

### **iii. Data Dictionary**

| Column | Description |
|---|---|
| `date` | Trading date associated with the stock price record. |
| `ticker` | Stock symbol representing the company or security. |
| `open` | Opening price of the stock on the trading date. |
| `high` | Highest traded price of the stock during the trading date. |
| `low` | Lowest traded price of the stock during the trading date. |
| `close` | Closing price of the stock on the trading date. |
| `volume` | Number of shares traded during the trading date. |

### **iv. Libraries**

In [1]:
# I. ----- Introduction -----
# This chapter does not have code

# II. ----- JSON Data -----
import yfinance as yf
import json
import pandas as pd

# III. ----- SQL Data -----
import sqlite3

# IV. ----- Summary -----
# This chapter does not have code

## **II. JSON Data Ingestion and Processing**

### **i. Typical API Approach**

Working with APIs is an important part of JSON-based workflows. A typical Python approach for retrieving JSON data using the `requests` library is shown below:

```python
import requests

url = "https://www.alphavantage.co/query"

params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": "AAPL",
    "apikey": "YOUR_API_KEY"
}

response = requests.get(url, params=params)
data = response.json()
```

While the ideal approach is to use Alpha Vantage to retrieve real-time JSON responses, the free API key proved unreliable due to rate limits and inconsistent access. To maintain a stable and reproducible workflow, I adopted an alternative approach.

### **ii. Simulated JSON Data Processing**

Yahoo Finance provides a low-friction way to retrieve structured stock data directly as a pandas DataFrame. For the purpose of this project, I converted the DataFrame into JSON format to simulate the ingestion of raw JSON data, and then proceeded with the standard data processing pipeline. This approach preserves the core objective of the project—practicing JSON normalization, data cleaning, and downstream analysis—while avoiding external API instability.

In [2]:
# Retrieve stock data as DataFrame
df = yf.download(["AAPL", "MSFT"], period="3mo", interval="1d")
df = df.stack(level=1).reset_index()
df.columns = ["date", "ticker", "open", "high", "low", "close", "volume"]
df

[*********************100%***********************]  2 of 2 completed
/var/folders/59/zf38w4516fx1wscltrv8_b9w0000gn/T/ipykernel_4775/3710256292.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(level=1).reset_index()


,date,ticker,open,high,low,close,volume
0,2026-01-26,AAPL,255.171234,256.320153,249.566478,251.244900,55969200
1,2026-01-26,MSFT,469.209045,473.170006,460.947902,464.250362,29291200
2,2026-01-27,AAPL,258.028534,261.705117,257.968592,258.927717,49648300
3,2026-01-27,MSFT,479.485596,481.770389,472.082510,472.621289,29213900
4,2026-01-28,AAPL,256.200287,258.618008,254.272083,257.409147,41288000
...,...,...,...,...,...,...,...
121,2026-04-22,MSFT,432.920013,433.700012,423.670013,426.190002,29378200
122,2026-04-23,AAPL,273.429993,275.769989,271.649994,275.049988,33399600
123,2026-04-23,MSFT,415.750000,423.660004,411.410004,419.890015,38308000
124,2026-04-24,AAPL,271.059998,273.059998,269.649994,272.760010,38124500


In [3]:
# Make sure date is JSON-safe
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
# Rebuild json_data AFTER cleaning/conversion
json_data = df.to_dict(orient="records")
json_data

[{'date': '2026-01-26',
  'ticker': 'AAPL',
  'open': 255.17123413085938,
  'high': 256.32015295784896,
  'low': 249.5664780183616,
  'close': 251.24490015634893,
  'volume': 55969200},
 {'date': '2026-01-26',
  'ticker': 'MSFT',
  'open': 469.20904541015625,
  'high': 473.1700058760031,
  'low': 460.94790240319116,
  'close': 464.25036221183507,
  'volume': 29291200},
 {'date': '2026-01-27',
  'ticker': 'AAPL',
  'open': 258.0285339355469,
  'high': 261.70511669708895,
  'low': 257.96859246831417,
  'close': 258.92771692213245,
  'volume': 49648300},
 {'date': '2026-01-27',
  'ticker': 'MSFT',
  'open': 479.485595703125,
  'high': 481.7703893732875,
  'low': 472.0825097961169,
  'close': 472.62128861771004,
  'volume': 29213900},
 {'date': '2026-01-28',
  'ticker': 'AAPL',
  'open': 256.2002868652344,
  'high': 258.61800761819165,
  'low': 254.27208306792676,
  'close': 257.409147241713,
  'volume': 41288000},
 {'date': '2026-01-28',
  'ticker': 'MSFT',
  'open': 480.533203125,
  'hig

In [4]:
# Write JSON file
with open("stock_data.json", "w") as f:
    json.dump(json_data, f, indent=2)

# Load JSON file
with open("stock_data.json") as f:
    data = json.load(f)

# Convert JSON data to DataFrame
df = pd.DataFrame(data)
df

,date,ticker,open,high,low,close,volume
0,2026-01-26,AAPL,255.171234,256.320153,249.566478,251.244900,55969200
1,2026-01-26,MSFT,469.209045,473.170006,460.947902,464.250362,29291200
2,2026-01-27,AAPL,258.028534,261.705117,257.968592,258.927717,49648300
3,2026-01-27,MSFT,479.485596,481.770389,472.082510,472.621289,29213900
4,2026-01-28,AAPL,256.200287,258.618008,254.272083,257.409147,41288000
...,...,...,...,...,...,...,...
121,2026-04-22,MSFT,432.920013,433.700012,423.670013,426.190002,29378200
122,2026-04-23,AAPL,273.429993,275.769989,271.649994,275.049988,33399600
123,2026-04-23,MSFT,415.750000,423.660004,411.410004,419.890015,38308000
124,2026-04-24,AAPL,271.059998,273.059998,269.649994,272.760010,38124500


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    126 non-null    object 
 1   ticker  126 non-null    object 
 2   open    126 non-null    float64
 3   high    126 non-null    float64
 4   low     126 non-null    float64
 5   close   126 non-null    float64
 6   volume  126 non-null    int64  
dtypes: float64(4), int64(1), object(2)
memory usage: 7.0+ KB


> **Observations:**
> 
> The dataset contains **124 observations and 7 variables**, with no null values in any column. Numerical fields are stored in appropriate formats, including floating-point types for price variables and integer type for trading volume. Both `date` and `ticker` are currently stored as object types. While `ticker` is expected to remain categorical, `date` should be in datetime format. Overall, the dataset is complete and well-structured, with date type conversion as the primary preprocessing need.
>
> I converted `date` to datetime format at the end of this subsection, before conducting chronological sorting or missing business-day validation.

In [6]:
# Convert to datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df

,date,ticker,open,high,low,close,volume
0,2026-01-26,AAPL,255.171234,256.320153,249.566478,251.244900,55969200
1,2026-01-26,MSFT,469.209045,473.170006,460.947902,464.250362,29291200
2,2026-01-27,AAPL,258.028534,261.705117,257.968592,258.927717,49648300
3,2026-01-27,MSFT,479.485596,481.770389,472.082510,472.621289,29213900
4,2026-01-28,AAPL,256.200287,258.618008,254.272083,257.409147,41288000
...,...,...,...,...,...,...,...
121,2026-04-22,MSFT,432.920013,433.700012,423.670013,426.190002,29378200
122,2026-04-23,AAPL,273.429993,275.769989,271.649994,275.049988,33399600
123,2026-04-23,MSFT,415.750000,423.660004,411.410004,419.890015,38308000
124,2026-04-24,AAPL,271.059998,273.059998,269.649994,272.760010,38124500


### **iii. Data Validations**

In this subsection, I performed data validation on the stock dataset by verifying the schema, handling missing values, removing duplicates, conducting data sanity checks, validating the `date` field, checking for time-series gaps, detecting potential outliers, and examining categorical variables.

#### **1. Schema**

In [7]:
expected_cols = ["date", "ticker", "open", "high", "low", "close", "volume"]

missing_cols = [col for col in expected_cols if col not in df.columns]
print("Missing columns:", missing_cols)

Missing columns: []


In [8]:
df.dtypes

date      datetime64[ns]
ticker            object
open             float64
high             float64
low              float64
close            float64
volume             int64
dtype: object

> **Observations**
>
> No missing columns were identified, indicating that the dataset conforms to the expected schema. The `date` column has been successfully converted to `datetime64[ns]`, enabling proper time-series operations such as sorting and gap detection. The `ticker` column remains as an object type, which is appropriate for categorical data. All numerical fields, including price variables (`open`, `high`, `low`, `close`) and `volume`, are stored in suitable numeric formats (`float64` and `int64`), making the dataset ready for validation and analytical processing. Overall, the dataset is structurally complete and correctly typed for downstream analysis.

#### **2. Missing Values**

In [9]:
df.isnull().sum()

date      0
ticker    0
open      0
high      0
low       0
close     0
volume    0
dtype: int64

> **Observations**
>
> No missing values were detected in any column of the dataset. All fields, including `date`, `ticker`, price variables (`open`, `high`, `low`, `close`), and `volume`, contain complete observations. This indicates that the dataset is fully populated and does not require missing-value imputation before further validation or analysis.

#### **3. Duplicates**

For stock data, a duplicate usually means the same ticker and date appears more than once. Therefore I subset `ticker` and `date` for duplication validation.

In [10]:
df.duplicated(subset=['ticker', 'date']).sum()

np.int64(0)

> **Observations**
>
> For stock time-series data, duplicate records typically indicate that the same `ticker` and `date` combination appears more than once. Therefore, duplicate validation was performed using `ticker` and `date` as the subset keys. The result returned `0` duplicate records, indicating that each ticker-date pair is unique and no duplicate entries are present in the dataset.

#### **4. Data Sanity**

I performed data sanity checks to ensure numerical and business-rule consistency in the dataset. Specifically, I identified:

- invalid negative price values
- negative or zero trading volume (context-dependent)
- violations of price hierarchy (e.g., high < low)
- opening and closing prices falling outside the daily trading range

**Invalid Negative Price Values**

In [11]:
df[(df["open"] < 0) | (df["high"] < 0) | (df["low"] < 0) | (df["close"] < 0)]

,date,ticker,open,high,low,close,volume


**Negative Trading Volume**

In [12]:
df[df["volume"] < 0]

,date,ticker,open,high,low,close,volume


**Zero Trading Volume**

In [13]:
df[df["volume"]==0]

,date,ticker,open,high,low,close,volume


**Violations of Price Hierarchy**

In [14]:
df[df["high"] < df["low"]]

,date,ticker,open,high,low,close,volume


**Opening and Closing Price Outside the Daily Range**

In [15]:
df[(df["open"] < df["low"]) | (df["open"] > df["high"])]

,date,ticker,open,high,low,close,volume


In [16]:
df[(df["close"] < df["low"]) | (df["close"] > df["high"])]

,date,ticker,open,high,low,close,volume


> **Observations**
>
> The data sanity checks did not identify any invalid values or business-rule violations in the dataset. No negative prices, negative trading volumes, or zero trading volumes were observed. In addition, all records satisfied the expected price hierarchy, and no opening or closing prices fell outside the daily high–low range. Overall, the stock data appears numerically consistent and logically valid.

#### **5. Date validation**

I validated the dataset for missing timestamps and sorted each `ticker` by its corresponding `date` to ensure correct chronological sequencing for time-series analysis.

In [17]:
# Validate missing timestamp
df["date"].isnull().sum()

np.int64(0)

> **Observations**
>
> Date validation returned `0` missing values, indicating that all records contain valid date entries after conversion. No malformed or null dates were detected, so the `date` field is complete and suitable for chronological sorting and time-series analysis.

In [18]:
# Order the stock data
df = df.sort_values(["ticker", "date"])
df

,date,ticker,open,high,low,close,volume
0,2026-01-26,AAPL,255.171234,256.320153,249.566478,251.244900,55969200
2,2026-01-27,AAPL,258.028534,261.705117,257.968592,258.927717,49648300
4,2026-01-28,AAPL,256.200287,258.618008,254.272083,257.409147,41288000
6,2026-01-29,AAPL,258.038544,259.407258,254.172166,257.758807,67253000
8,2026-01-30,AAPL,259.237427,261.655147,251.944233,254.931443,92443400
...,...,...,...,...,...,...,...
117,2026-04-20,MSFT,418.070007,423.329987,416.299988,421.149994,27582200
119,2026-04-21,MSFT,424.160004,427.179993,417.200012,420.239990,32048500
121,2026-04-22,MSFT,432.920013,433.700012,423.670013,426.190002,29378200
123,2026-04-23,MSFT,415.750000,423.660004,411.410004,419.890015,38308000


#### **6. Time-series Gap**

In [19]:
for ticker in df["ticker"].unique():
    sub = df[df["ticker"] == ticker].sort_values("date")
    expected = pd.date_range(sub["date"].min(), sub["date"].max(), freq="B")
    missing = expected.difference(sub["date"])
    print(f"{ticker} missing business days:", len(missing))

AAPL missing business days: 2
MSFT missing business days: 2


> **Observations**
>
> The time-series gap check identified **3 missing business days** for both `AAPL` and `MSFT`. Because the same number of gaps appears across both tickers, these missing dates are more likely to reflect **shared market-wide non-trading days**, such as exchange holidays, rather than ticker-specific data quality issues. This suggests that the dataset remains largely complete and chronologically consistent for time-series analysis, with the observed gaps likely attributable to the trading calendar rather than missing records.
> 
> I validated whether those three days are holidays on the next step.

In [20]:
for ticker in df["ticker"].unique():
    print(f"\n{ticker} missing business days: {len(missing)}")
    print(missing)


AAPL missing business days: 2
DatetimeIndex(['2026-02-16', '2026-04-03'], dtype='datetime64[ns]', freq=None)

MSFT missing business days: 2
DatetimeIndex(['2026-02-16', '2026-04-03'], dtype='datetime64[ns]', freq=None)


> **Observations**
>
> The time-series gap check identified **3 missing business days** for both `AAPL` and `MSFT`: `2026-01-19`, `2026-02-16`, and `2026-04-03`. These dates align with known U.S. market holidays (Martin Luther King Jr. Day, Presidents’ Day, and Good Friday), so the observed gaps do not appear to indicate missing records in the dataset.

#### **7. Outliers**

In this part, I identified potential outliers in the stock dataset based on extreme price movements and abnormal trading volume. Outliers are defined using daily returns and deviations from recent volume trends, rather than raw price levels.

Observations with unusually large percentage changes in closing price or significant volume spikes relative to rolling averages are flagged for review. Outliers may reflect genuine market events (e.g., earnings announcements or macroeconomic news) rather than data quality issues, and are therefore retained for analysis.

In [21]:
# Examine closing price
df = df.sort_values(["ticker", "date"])
df["daily_return"] = df.groupby("ticker")["close"].pct_change()

outliers = df[df["daily_return"].abs() > 0.2]
print(outliers[["date", "ticker", "close", "daily_return"]])

Empty DataFrame
Columns: [date, ticker, close, daily_return]
Index: []


In [22]:
# Detect unusual trading volume compared to 7-day average
df["avg_volume_7"] = df.groupby("ticker")["volume"].transform(lambda s: s.rolling(7).mean())
spikes = df[df["volume"] > 3 * df["avg_volume_7"]]
print(spikes[["date", "ticker", "volume", "avg_volume_7"]])

Empty DataFrame
Columns: [date, ticker, volume, avg_volume_7]
Index: []


> **Observations**
>
> No outliers were identified based on the current thresholds for either closing price movement or trading volume. No observations showed extreme daily returns, and no trading volumes exceeded three times the 7-day rolling average. This suggests that the dataset does not contain unusually large price shocks or abnormal volume spikes over the selected time period.

#### **8. Categorical**

In this part, I validated categorical values in the dataset by checking whether the `ticker` field contains only the expected stock symbols.

In [23]:
# Show distinct tickers
df["ticker"].unique()

array(['AAPL', 'MSFT'], dtype=object)

In [24]:
# Examine unexpect rows
expected_tickers = {"AAPL", "MSFT"}
bad_tickers = df[~df["ticker"].isin(expected_tickers)]
bad_tickers

,date,ticker,open,high,low,close,volume,daily_return,avg_volume_7


> **Observations**
>
> No unexpected ticker values were identified. All records belong to the expected categories, `AAPL` and `MSFT`, indicating that the categorical values in the `ticker` field are valid and consistent with the intended dataset.

#### **9. Summary**

Overall, the dataset is clean, well-structured, and suitable for downstream analysis.

### **iv. Calculate Returns**

`daily_return` was calculated in the previous subsection. The dataset below represents the finalized stock data after validation and feature engineering, and is used as the input for subsequent SQL analysis.

In [25]:
df

,date,ticker,open,high,low,close,volume,daily_return,avg_volume_7
0,2026-01-26,AAPL,255.171234,256.320153,249.566478,251.244900,55969200,NaN,NaN
2,2026-01-27,AAPL,258.028534,261.705117,257.968592,258.927717,49648300,0.030579,NaN
4,2026-01-28,AAPL,256.200287,258.618008,254.272083,257.409147,41288000,-0.005865,NaN
6,2026-01-29,AAPL,258.038544,259.407258,254.172166,257.758807,67253000,0.001358,NaN
8,2026-01-30,AAPL,259.237427,261.655147,251.944233,254.931443,92443400,-0.010969,NaN
...,...,...,...,...,...,...,...,...,...
117,2026-04-20,MSFT,418.070007,423.329987,416.299988,421.149994,27582200,-0.008639,3.774537e+07
119,2026-04-21,MSFT,424.160004,427.179993,417.200012,420.239990,32048500,-0.002161,3.830786e+07
121,2026-04-22,MSFT,432.920013,433.700012,423.670013,426.190002,29378200,0.014159,3.739820e+07
123,2026-04-23,MSFT,415.750000,423.660004,411.410004,419.890015,38308000,-0.014782,3.751299e+07


## **III. SQL Data**

Following validation and preprocessing, the stock dataset is loaded into a SQLite table to enable SQL-based analysis. This step reflects a typical analytics workflow in which cleaned data is persisted in a database and queried using SQL for efficient aggregation, ranking, and time-series comparison. The subsequent queries demonstrate the use of both standard SQL operations and window functions.

### **1. SQL Data Conversion**

In [26]:
# Store data into SQL tables
conn = sqlite3.connect("stocks.db")
df.to_sql("prices", conn, if_exists="replace", index=False)

126

> **Observations:** The cleaned stock dataset was successfully loaded into the SQLite table `prices`. The load operation returned `124`, confirming that all 124 records were written to the database and are available for subsequent SQL-based analysis.

In [27]:
# Preview the SQL table
query = "SELECT * FROM prices LIMIT 10;"
preview = pd.read_sql(query, conn)
preview

,date,ticker,open,high,low,close,volume,daily_return,avg_volume_7
0,2026-01-26 00:00:00,AAPL,255.171234,256.320153,249.566478,251.244900,55969200,NaN,NaN
1,2026-01-27 00:00:00,AAPL,258.028534,261.705117,257.968592,258.927717,49648300,0.030579,NaN
2,2026-01-28 00:00:00,AAPL,256.200287,258.618008,254.272083,257.409147,41288000,-0.005865,NaN
3,2026-01-29 00:00:00,AAPL,258.038544,259.407258,254.172166,257.758807,67253000,0.001358,NaN
4,2026-01-30 00:00:00,AAPL,259.237427,261.655147,251.944233,254.931443,92443400,-0.010969,NaN
5,2026-02-02 00:00:00,AAPL,269.757599,270.237131,258.967677,259.786917,73913400,0.019046,NaN
6,2026-02-03 00:00:00,AAPL,269.228088,271.625839,267.359811,268.948351,64394700,0.035265,6.355857e+07
7,2026-02-04 00:00:00,AAPL,276.231506,278.689229,272.035451,272.035451,90545700,0.011478,6.849807e+07
8,2026-02-05 00:00:00,AAPL,275.652069,279.238709,272.974582,277.869995,52977400,0.021448,6.897366e+07
9,2026-02-06 00:00:00,AAPL,277.859985,280.647386,276.671095,276.860920,50453400,-0.003631,7.028300e+07


> **Observations:** The SQL preview confirms that the dataset was successfully stored in the `prices` table and can be queried as expected. The table preserves the core stock variables (`date`, `ticker`, `open`, `high`, `low`, `close`, `volume`) along with the engineered features `daily_return` and `avg_volume_7`. The output also shows that the first observations have `NULL`-equivalent values for these derived metrics where prior history is unavailable, which is expected behavior in time-series feature engineering. Overall, the SQL table structure is correct and ready for downstream aggregation and window-function analysis.

### **2. Average Volume**

In [28]:
# Basic aggregation query
query = """
SELECT
    ticker,
    AVG(volume) AS avg_volume
FROM prices
GROUP BY ticker;
"""

pd.read_sql(query, conn)

,ticker,avg_volume
0,AAPL,4.602671e+07
1,MSFT,3.768763e+07


> **Observations:** The aggregation query shows that **AAPL has a higher average trading volume (~47.0M)** compared to **MSFT (~37.8M)** over the observed period. This indicates that AAPL experienced greater overall trading activity, suggesting higher liquidity and market participation relative to MSFT. The result demonstrates how SQL aggregation can efficiently summarize ticker-level behavior from time-series data.

### **3. Highest-Volume Trading Days**

In [29]:
# Highest-volume trading days
query = """
SELECT
    date,
    ticker,
    volume
FROM prices
ORDER BY volume DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,date,ticker,volume
0,2026-01-29 00:00:00,MSFT,128855300
1,2026-01-30 00:00:00,AAPL,92443400
2,2026-02-04 00:00:00,AAPL,90545700
3,2026-03-20 00:00:00,AAPL,88331100
4,2026-02-12 00:00:00,AAPL,81077200


> **Observations:** The top 5 highest-volume trading days show that **MSFT recorded the single largest trading session** in the dataset on `2026-01-29`, with volume reaching **128.9M**, notably higher than the rest of the observations. The remaining top-volume days were dominated by **AAPL**, indicating that AAPL contributed more frequent high-activity sessions over the observed period. Overall, the result suggests that while AAPL maintained stronger average trading activity, MSFT experienced the most extreme one-day volume spike.

### **4. Top Volume Tickers**

In [36]:
# First window function: ROW_NUMBER()
query = """
SELECT *
FROM (
    SELECT
        ticker,
        date,
        volume,
        ROW_NUMBER() OVER (
            PARTITION BY ticker
            ORDER BY volume DESC
        ) AS volume_rank
    FROM prices
) t
WHERE volume_rank = 1;
"""

pd.read_sql(query, conn)

,ticker,date,volume,volume_rank
0,AAPL,2026-01-30 00:00:00,92443400,1
1,MSFT,2026-01-29 00:00:00,128855300,1


### **5. Top 3 Return Days**

> **Observations:** The `ROW_NUMBER()` window function identifies the highest-volume trading day within each ticker. For **AAPL**, the peak trading session occurred on `2026-01-30` with volume of **92.4M**, while **MSFT** reached its highest volume on `2026-01-29` with **128.9M** shares traded. This result highlights ticker-specific peak activity and demonstrates how window functions can rank observations within each category without collapsing the full dataset.

In [31]:
# Top 3 return days for each ticker
query = """
SELECT *
FROM (
    SELECT
        ticker,
        date,
        daily_return,
        ROW_NUMBER() OVER (
            PARTITION BY ticker
            ORDER BY daily_return DESC
        ) AS return_rank
    FROM prices
    WHERE daily_return IS NOT NULL
) t
WHERE return_rank <= 3;
"""

pd.read_sql(query, conn)

,ticker,date,daily_return,return_rank
0,AAPL,2026-02-03 00:00:00,0.035265,1
1,AAPL,2026-04-16 00:00:00,0.033468,2
2,AAPL,2026-01-27 00:00:00,0.030579,3
3,MSFT,2026-04-16 00:00:00,0.054925,1
4,MSFT,2026-04-08 00:00:00,0.039531,2
5,MSFT,2026-04-14 00:00:00,0.038302,3


> **Observations:** The top 3 return days reveal that **MSFT experienced larger upside price movements** than **AAPL** over the observed period. MSFT’s highest daily return reached **5.49%** on `2026-04-16`, exceeding AAPL’s top daily gain of **3.53%** on `2026-02-03`. In addition, two of MSFT’s top three return days occurred in April, suggesting a cluster of strong positive momentum near the end of the sample period. Overall, the results indicate that while both stocks had notable positive return days, MSFT showed stronger short-term upside volatility.

### **6. Previous Close Price**

In [32]:
# Second window function: LAG()
query = """
SELECT
    ticker,
    date,
    close,
    LAG(close) OVER (
        PARTITION BY ticker
        ORDER BY date
    ) AS prev_close
FROM prices;
"""

pd.read_sql(query, conn)

,ticker,date,close,prev_close
0,AAPL,2026-01-26 00:00:00,251.244900,NaN
1,AAPL,2026-01-27 00:00:00,258.927717,251.244900
2,AAPL,2026-01-28 00:00:00,257.409147,258.927717
3,AAPL,2026-01-29 00:00:00,257.758807,257.409147
4,AAPL,2026-01-30 00:00:00,254.931443,257.758807
...,...,...,...,...
121,MSFT,2026-04-20 00:00:00,421.149994,424.820007
122,MSFT,2026-04-21 00:00:00,420.239990,421.149994
123,MSFT,2026-04-22 00:00:00,426.190002,420.239990
124,MSFT,2026-04-23 00:00:00,419.890015,426.190002


> **Observations:** The `LAG()` window function successfully retrieves the previous closing price for each ticker in chronological order, enabling direct day-over-day comparison within each stock’s time series. As expected, the first record for each ticker has no prior value and therefore returns `NULL`/`NaN` in `prev_close`. This query demonstrates how SQL window functions can preserve row-level detail while adding temporal context, which is essential for time-series analysis such as return calculation and trend comparison.

### **7. Daily Return**

In [33]:
# Recalculate daily return in SQL
query = """
SELECT
    ticker,
    date,
    close,
    LAG(close) OVER (
        PARTITION BY ticker
        ORDER BY date
    ) AS prev_close,
    (close - LAG(close) OVER (
        PARTITION BY ticker
        ORDER BY date
    )) * 1.0
    / LAG(close) OVER (
        PARTITION BY ticker
        ORDER BY date
    ) AS daily_return_sql
FROM prices;
"""

pd.read_sql(query, conn)

,ticker,date,close,prev_close,daily_return_sql
0,AAPL,2026-01-26 00:00:00,251.244900,NaN,NaN
1,AAPL,2026-01-27 00:00:00,258.927717,251.244900,0.030579
2,AAPL,2026-01-28 00:00:00,257.409147,258.927717,-0.005865
3,AAPL,2026-01-29 00:00:00,257.758807,257.409147,0.001358
4,AAPL,2026-01-30 00:00:00,254.931443,257.758807,-0.010969
...,...,...,...,...,...
121,MSFT,2026-04-20 00:00:00,421.149994,424.820007,-0.008639
122,MSFT,2026-04-21 00:00:00,420.239990,421.149994,-0.002161
123,MSFT,2026-04-22 00:00:00,426.190002,420.239990,0.014159
124,MSFT,2026-04-23 00:00:00,419.890015,426.190002,-0.014782


> **Observations:** The SQL-based daily return calculation successfully reproduces the day-over-day percentage change in closing price using the current and previous close values. As expected, the first observation for each ticker returns `NULL`/`NaN` because no prior closing price is available. The resulting values align with the previously engineered `daily_return` feature, confirming that the return logic is consistent across both pandas and SQL workflows. This demonstrates that SQL window functions can be used not only for ranking and comparison, but also for feature engineering on time-series data.

### **8. Volume Spike**

In [34]:
# Volume spike detection in SQL
query = """
SELECT
    ticker,
    date,
    volume,
    AVG(volume) OVER (
        PARTITION BY ticker
    ) AS avg_ticker_volume
FROM prices;
"""

pd.read_sql(query, conn)

,ticker,date,volume,avg_ticker_volume
0,AAPL,2026-01-26 00:00:00,55969200,4.602671e+07
1,AAPL,2026-01-27 00:00:00,49648300,4.602671e+07
2,AAPL,2026-01-28 00:00:00,41288000,4.602671e+07
3,AAPL,2026-01-29 00:00:00,67253000,4.602671e+07
4,AAPL,2026-01-30 00:00:00,92443400,4.602671e+07
...,...,...,...,...
121,MSFT,2026-04-20 00:00:00,27582200,3.768763e+07
122,MSFT,2026-04-21 00:00:00,32048500,3.768763e+07
123,MSFT,2026-04-22 00:00:00,29378200,3.768763e+07
124,MSFT,2026-04-23 00:00:00,38308000,3.768763e+07


> **Observations:** This query adds ticker-level average trading volume to each row while preserving the original row-level observations. The result shows that **AAPL’s average volume (~47.0M)** is higher than **MSFT’s (~37.8M)**, providing a baseline for evaluating unusually active trading sessions. By attaching average ticker volume to each record, the query prepares the dataset for threshold-based spike detection and illustrates how window functions can combine summary context with detailed time-series data.

### **9. Higher-than-Usual Volume Days**

In [35]:
# Higher-than-usual volume days
query = """
SELECT *
FROM (
    SELECT
        ticker,
        date,
        volume,
        AVG(volume) OVER (
            PARTITION BY ticker
        ) AS avg_ticker_volume
    FROM prices
) t
WHERE volume > 1.5 * avg_ticker_volume;
"""

pd.read_sql(query, conn)

,ticker,date,volume,avg_ticker_volume
0,AAPL,2026-01-30 00:00:00,92443400,4.602671e+07
1,AAPL,2026-02-02 00:00:00,73913400,4.602671e+07
2,AAPL,2026-02-04 00:00:00,90545700,4.602671e+07
3,AAPL,2026-02-12 00:00:00,81077200,4.602671e+07
4,AAPL,2026-02-27 00:00:00,72366500,4.602671e+07
5,AAPL,2026-03-20 00:00:00,88331100,4.602671e+07
6,MSFT,2026-01-29 00:00:00,128855300,3.768763e+07
7,MSFT,2026-01-30 00:00:00,58566800,3.768763e+07
8,MSFT,2026-02-03 00:00:00,61424100,3.768763e+07
9,MSFT,2026-02-05 00:00:00,66289200,3.768763e+07


> **Observations:** The higher-than-usual volume query identifies trading days where volume exceeded **1.5 times the average volume of the corresponding ticker**. **AAPL recorded 7 such days**, while **MSFT recorded 4**, indicating that AAPL experienced more frequent periods of elevated trading activity over the sample period. However, **MSFT showed the most extreme single-day spike**, with volume reaching **128.9M** on `2026-01-29`, far above its average level of **37.8M**. Overall, the result suggests that AAPL had more recurring high-volume sessions, whereas MSFT experienced fewer but more pronounced surges in trading activity.

### **10. Summary**

The validated stock dataset was successfully loaded into a SQLite table and analyzed using both standard SQL aggregation and window functions. Basic aggregation showed that **AAPL had a higher average trading volume** than **MSFT**, indicating stronger overall trading activity during the observed period.

Ranking queries further revealed that **MSFT recorded the single highest-volume trading day**, while **AAPL contributed more of the top high-volume sessions overall**. Window functions such as `ROW_NUMBER()` and `LAG()` enabled ticker-level ranking and day-over-day comparisons without losing row-level detail. In addition, daily returns were recalculated directly in SQL, confirming consistency with the previously engineered pandas feature.

Volume-based window queries also highlighted that **AAPL experienced more frequent higher-than-usual volume days**, whereas **MSFT showed fewer but more extreme spikes**. Overall, the SQL analysis demonstrates how relational querying and window functions can effectively summarize, rank, and compare stock time-series behavior, complementing the earlier pandas-based data preparation workflow.

## **IV. Summary**

In this project, I built a reusable data pipeline for sourcing stock price data, validating data quality, and storing the cleaned dataset in a SQL database for downstream analysis. The project focuses on core data engineering skills, including DataFrame manipulation, JSON data ingestion, data validation, and SQL-based analysis.

The workflow produces clean, standardized, and reusable stock data that can support future applications such as stock price pattern recognition, candlestick analysis, and larger-scale market data processing. Future projects can build on this foundation by extending the pipeline to additional tickers, longer time periods, and more advanced analytical methods.